# Cyclostationary Analysis Demo

Demonstrates `CyclostationaryAnalyzer` on two test signals:
- Rectangular-pulse BPSK
- Bandlimited Gaussian noise modulating a cosine

Adapted from Fabio Casagrande Hirono's [cyclostationarity_analysis](https://github.com/fchirono/cyclostationarity_analysis) (November 2022), which draws from [Chad Spooner's Cyclostationary Signal Processing blog](https://cyclostationary.blog/) and W. A. Gardner, "Exploitation of Spectral Redundancy in Cyclostationary Signals," *IEEE Signal Processing Magazine*, vol. 8, no. 2, pp. 14–36, 1991.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

from cyclo import CyclostationaryAnalyzer
import cyclo.signals as sig

In [ ]:
def polygon_under_graph(x, y):
    """Vertex list for filling under a (x, y) line graph (x must be ascending).
    From https://matplotlib.org/stable/gallery/mplot3d/polys3d.html
    """
    return [(x[0], 0.), *zip(x, y), (x[-1], 0.)]

## 1. Generate test signal

Switch `signal` between `'bpsk'` and `'lowpassmod_cos'` to try both cases.

In [ ]:
signal = 'bpsk'
# signal = 'lowpassmod_cos'

fs = 1.0
signal_power_dB = 0.0
noise_power_dB = -10.0

rng = np.random.default_rng(42)

if signal == 'bpsk':
    T_bits = 10
    num_bits = 32768
    fc = 0.05
    y = sig.create_rect_bpsk(T_bits, num_bits, fc, signal_power_dB, rng=rng)

elif signal == 'lowpassmod_cos':
    N_samples = 327680
    N_filter = 6
    fc_filter = 0.05 * fs
    f_cos = 0.15 * fs
    y = sig.create_lowpassmod_cos(N_samples, N_filter, fc_filter, f_cos, fs,
                                  signal_power_dB, rng=rng)

n = sig.create_noise(y, SNR_dB=signal_power_dB - noise_power_dB, rng=rng)
noisy_y = y + n

Py, Pn = np.var(y), np.var(n)
print(f'Desired SNR  = {signal_power_dB - noise_power_dB:.1f} dB')
print(f'Resulting SNR = {10*np.log10(Py/Pn):.1f} dB')

In [ ]:
N_plot = 100
t_plot = np.arange(N_plot)

fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0].plot(t_plot, y[:N_plot].real, label='Re(y)')
axes[0].plot(t_plot, y[:N_plot].imag, '--', label='Im(y)')
axes[0].set_title('Original vs. noisy time-domain signal', fontsize=18)
axes[0].set_ylabel('Amplitude', fontsize=15)
axes[0].legend(loc='lower right', fontsize=12)

axes[1].plot(t_plot, noisy_y[:N_plot].real, label='Re(noisy y)')
axes[1].plot(t_plot, noisy_y[:N_plot].imag, '--', label='Im(noisy y)')
axes[1].set_xlabel('Samples', fontsize=15)
axes[1].set_ylabel('Amplitude', fontsize=15)
axes[1].legend(loc='lower right', fontsize=12)
plt.tight_layout()

## 2. Non-conjugate cyclic periodogram

In [ ]:
N_psd = 128
N_alpha = 11
alpha_vec = np.linspace(0., 1., N_alpha) * fs

analyzer = CyclostationaryAnalyzer(noisy_y, fs=fs)
freq_vec, Syy, rho_y = analyzer.cyclic_periodogram(alpha_vec, N_psd)

T = y.shape[0] / fs
freq_res = fs / N_psd
print(f'Time-bandwidth product = {T * freq_res:.2f}')

In [ ]:
lines = ['-', '--', '-.', ':']
linewidths = [2.5, 1, 0.5, 0.25]
N_lines = len(lines)

max_Syy_dB = 10 * np.log10(np.nanmax(np.abs(Syy)))

plt.figure(figsize=(9, 6))
for a in range(N_alpha // 2 + 1):
    plt.plot(freq_vec / fs,
             10 * np.log10(np.abs(Syy[a, :])),
             linestyle=lines[a % N_lines], linewidth=linewidths[a // N_lines],
             label=r'$\alpha/f_s$={:.2f}'.format(alpha_vec[a] / fs))
plt.legend(loc='upper left', fontsize=12)
plt.xlabel(r'freq/$f_s$', fontsize=12)
plt.xlim([-0.5, 0.5])
plt.ylabel('Magnitude [dB]', fontsize=12)
plt.ylim([max_Syy_dB - 25, max_Syy_dB + 5])
plt.title('Spectral correlation density', fontsize=15)
plt.grid()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(9, 6))
for a in range(N_alpha // 2 + 1):
    plt.plot(freq_vec / fs,
             np.abs(rho_y[a, :]),
             linestyle=lines[a % N_lines], linewidth=linewidths[a // N_lines],
             label=r'$\alpha/f_s$={:.2f}'.format(alpha_vec[a] / fs))
plt.legend(loc='upper left', fontsize=12)
plt.xlabel(r'freq/$f_s$', fontsize=12)
plt.xlim([-0.5, 0.5])
plt.ylabel('Magnitude [Linear]', fontsize=12)
plt.ylim([0., 1.2])
plt.title('Spectral coherence function', fontsize=15)
plt.grid()
plt.tight_layout()

## 3. 3D plots

In [ ]:
freqs_inside = np.zeros((N_alpha, N_psd), dtype=bool)
for a, alpha in enumerate(alpha_vec):
    freqs_inside[a, :] = np.abs(freq_vec) <= (fs - np.abs(alpha)) / 2

cmap = plt.colormaps['inferno']
facecolors = cmap(np.linspace(0, 1, N_alpha + 2))[:N_alpha]
transparency = 0.8

# --- Spectral correlation density ---
fig1 = plt.figure(figsize=(9, 6))
ax1 = fig1.add_subplot(projection='3d')
verts = [polygon_under_graph(freq_vec[freqs_inside[a]] / fs,
                              np.abs(Syy[a, freqs_inside[a]]))
         for a in range(N_alpha)]
poly1 = PolyCollection(verts[::-1], facecolors=facecolors, alpha=transparency,
                        edgecolors='k', linewidths=0.75)
ax1.add_collection3d(poly1, zs=alpha_vec[::-1], zdir='y')
ax1.set(xlim=(-0.5, 0.5), ylim=(alpha_vec[-1], alpha_vec[0]),
        zlim=(0., np.nanmax(np.abs(Syy))))
ax1.set_xlabel(r'freq/$f_s$', fontsize=12)
ax1.set_ylabel(r'$\alpha/f_s$', fontsize=12)
ax1.set_zlabel('Magnitude [Linear]', fontsize=12)
ax1.set_title('Spectral correlation density', fontsize=15)

# --- Spectral coherence ---
fig2 = plt.figure(figsize=(9, 6))
ax2 = fig2.add_subplot(projection='3d')
verts2 = [polygon_under_graph(freq_vec[freqs_inside[a]] / fs,
                               np.abs(rho_y[a, freqs_inside[a]]))
          for a in range(N_alpha)]
poly2 = PolyCollection(verts2[::-1], facecolors=facecolors, alpha=transparency,
                        edgecolors='k', linewidths=0.75)
ax2.add_collection3d(poly2, zs=alpha_vec[::-1], zdir='y')
ax2.set(xlim=(-0.5, 0.5), ylim=(alpha_vec[-1], alpha_vec[0]), zlim=(0., 1.))
ax2.set_xlabel(r'freq/$f_s$', fontsize=12)
ax2.set_ylabel(r'$\alpha/f_s$', fontsize=12)
ax2.set_zlabel('Magnitude [Linear]', fontsize=12)
ax2.set_title('Spectral coherence', fontsize=15)
ax2.view_init(elev=30, azim=-45)

## 4. Conjugate cyclic periodogram

In [ ]:
# Conjugate alpha range is centered on 2*fc with spacing 1/T_bits
# (only relevant for BPSK; adjust manually for other signals)
if signal == 'bpsk':
    N_alpha_c = 19
    alpha_vec_c = 2 * fc + np.linspace(-N_alpha_c // 2, N_alpha_c // 2 - 1, N_alpha_c) * (1 / T_bits)
else:
    N_alpha_c = 19
    alpha_vec_c = np.linspace(-0.5, 0.5, N_alpha_c) * fs

freq_vec_c, Syy_c, rho_y_c = analyzer.cyclic_periodogram(alpha_vec_c, N_psd, mode='conj')

In [ ]:
max_Syyc_dB = 10 * np.log10(np.nanmax(np.abs(Syy_c)))

plt.figure(figsize=(9, 6))
for a in range(7, N_alpha_c - 7):
    plt.plot(freq_vec_c / fs,
             10 * np.log10(np.abs(Syy_c[a, :])),
             linestyle=lines[(a - 7) % N_lines],
             linewidth=linewidths[(a - 7) // N_lines],
             label=r'$\alpha/f_s$={:.2f}'.format(alpha_vec_c[a] / fs))
plt.legend(loc='upper left', fontsize=12)
plt.xlabel(r'freq/$f_s$', fontsize=12)
plt.xlim([-0.5, 0.5])
plt.ylabel('Magnitude [dB]', fontsize=12)
plt.ylim([max_Syyc_dB - 25, max_Syyc_dB + 5])
plt.title('Conjugate spectral correlation density', fontsize=15)
plt.grid()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(9, 6))
for a in range(7, N_alpha_c - 7):
    plt.plot(freq_vec_c / fs,
             np.abs(rho_y_c[a, :]),
             linestyle=lines[(a - 7) % N_lines],
             linewidth=linewidths[(a - 7) // N_lines],
             label=r'$\alpha/f_s$={:.2f}'.format(alpha_vec_c[a] / fs))
plt.legend(loc='upper left', fontsize=12)
plt.xlabel(r'freq/$f_s$', fontsize=12)
plt.xlim([-0.5, 0.5])
plt.ylabel('Magnitude [Linear]', fontsize=12)
plt.ylim([0., 1.2])
plt.title('Conjugate spectral coherence function', fontsize=15)
plt.grid()
plt.tight_layout()

In [ ]:
freqs_inside_c = np.zeros((N_alpha_c, N_psd), dtype=bool)
for a, alpha in enumerate(alpha_vec_c):
    freqs_inside_c[a, :] = np.abs(freq_vec_c) <= (fs - np.abs(alpha)) / 2

facecolors_c = cmap(np.linspace(0, 1, N_alpha_c + 2))[:N_alpha_c]

# --- Conjugate spectral correlation density ---
fig3 = plt.figure(figsize=(12, 9))
ax3 = fig3.add_subplot(projection='3d')
verts_c = [polygon_under_graph(freq_vec_c[freqs_inside_c[a]] / fs,
                                np.abs(Syy_c[a, freqs_inside_c[a]]))
           for a in range(N_alpha_c)]
poly3 = PolyCollection(verts_c[::-1], facecolors=facecolors_c, alpha=transparency,
                        edgecolors='k', linewidths=0.75)
ax3.add_collection3d(poly3, zs=alpha_vec_c[::-1], zdir='y')
ax3.set(xlim=(-0.5, 0.5), ylim=(alpha_vec_c[-1], alpha_vec_c[0]),
        zlim=(0., np.nanmax(np.abs(Syy_c))))
ax3.set_xlabel(r'freq/$f_s$', fontsize=12)
ax3.set_ylabel(r'$\alpha/f_s$', fontsize=12)
ax3.set_zlabel('Magnitude [Linear]', fontsize=12)
ax3.set_title('Conjugate spectral correlation density', fontsize=15)

# --- Conjugate spectral coherence ---
fig4 = plt.figure(figsize=(12, 9))
ax4 = fig4.add_subplot(projection='3d')
verts4 = [polygon_under_graph(freq_vec_c[freqs_inside_c[a]] / fs,
                               np.abs(rho_y_c[a, freqs_inside_c[a]]))
          for a in range(N_alpha_c)]
poly4 = PolyCollection(verts4[::-1], facecolors=facecolors_c, alpha=transparency,
                        edgecolors='k', linewidths=0.75)
ax4.add_collection3d(poly4, zs=alpha_vec_c[::-1], zdir='y')
ax4.set(xlim=(-0.5, 0.5), ylim=(alpha_vec_c[-1], alpha_vec_c[0]), zlim=(0., 1.))
ax4.set_xlabel(r'freq/$f_s$', fontsize=12)
ax4.set_ylabel(r'$\alpha/f_s$', fontsize=12)
ax4.set_zlabel('Magnitude [Linear]', fontsize=12)
ax4.set_title('Conjugate spectral coherence', fontsize=15)